In [10]:
import polars as pl
from pathlib import Path

DATA_PATH = Path("../data/raw/qna.csv")

print("Dataset:", DATA_PATH)
print("Exists:", DATA_PATH.exists())
print("Size (GB):", round(DATA_PATH.stat().st_size / (1024**3), 2))

Dataset: ../data/raw/qna.csv
Exists: True
Size (GB): 2.64


In [11]:
df = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    ignore_errors=False
)

df

In [12]:
schema = df.collect_schema()

print("Jumlah kolom:", len(schema))
print("\nKolom dan tipe data:")

for name, dtype in schema.items():
    print(f"- {name}: {dtype}")

Jumlah kolom: 10

Kolom dan tipe data:
- id: String
- title: String
- user: String
- question: String
- question_clean: String
- question_post_date: String
- answer_by: String
- answer_post_date: String
- answer: String
- answer_clean: String


In [13]:
row_count = (
    df
    .select(pl.len().alias("row_count"))
    .collect()
)

row_count

row_count
u32
681203


In [14]:
columns = schema.names()

missing = (
    df
    .select([
        pl.col(column)
        .null_count()
        .alias(column)
        for column in columns
    ])
    .collect()
)

missing

id,title,user,question,question_clean,question_post_date,answer_by,answer_post_date,answer,answer_clean
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,9,4,22,97,0,0,0,0,15


In [15]:
missing_summary = (
    df
    .select([
        pl.col(column).null_count().alias(f"{column}_missing")
        for column in columns
    ])
    .collect()
)

missing_summary

id_missing,title_missing,user_missing,question_missing,question_clean_missing,question_post_date_missing,answer_by_missing,answer_post_date_missing,answer_missing,answer_clean_missing
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,9,4,22,97,0,0,0,0,15


In [16]:
duplicate_id = (
    df
    .group_by("id")
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
    .collect()
)

duplicate_id.head(20)

id,len
str,u32
"""576affb8abcdce1d710181f5""",19
"""576affb8abcdce1d710181b7""",12
"""576affdbabcdce1d7101de1c""",12
"""576affb8abcdce1d71018471""",11
"""576affb8abcdce1d710180c4""",11
…,…
"""576affb8abcdce1d7101838d""",9
"""576affb8abcdce1d71017f38""",9
"""576affe7abcdce1d710203ab""",9


In [17]:
print("Jumlah ID yang duplikat:", duplicate_id.height)

Jumlah ID yang duplikat: 212182


In [18]:
text_profile = (
    df
    .select([
        pl.col("question")
        .str.len_chars()
        .alias("question_length"),

        pl.col("answer")
        .str.len_chars()
        .alias("answer_length")
    ])
    .select([
        pl.col("question_length").mean().alias("question_mean"),
        pl.col("question_length").median().alias("question_median"),
        pl.col("question_length").min().alias("question_min"),
        pl.col("question_length").max().alias("question_max"),

        pl.col("answer_length").mean().alias("answer_mean"),
        pl.col("answer_length").median().alias("answer_median"),
        pl.col("answer_length").min().alias("answer_min"),
        pl.col("answer_length").max().alias("answer_max"),
    ])
    .collect()
)

text_profile

question_mean,question_median,question_min,question_max,answer_mean,answer_median,answer_min,answer_max
f64,f64,u32,u32,f64,f64,u32,u32
364.538255,294.0,1,232868,1722.018789,1624.0,2,471356


In [19]:
sample = (
    df
    .select([
        "id",
        "title",
        "question_post_date",
        "answer_post_date",
        "question",
        "answer"
    ])
    .head(5)
    .collect()
)

sample

id,title,question_post_date,answer_post_date,question,answer
str,str,str,str,str,str
"""64b0ea5657ecf6002526920e""","""Ciri-ciri penyakit ablasio ret…","""Fri Jul 14 2023 13:25:00 GMT+0…","""Fri Jul 14 2023 17:59:00 GMT+0…","""<p>Pandangan mata saya udah se…",""" <p>Alo, terimakasih atas per…"
"""630c1595666c1406f7a2798f""","""Apakah ablasi retina bisa semb…","""Mon Aug 29 2022 08:25:00 GMT+0…","""Mon Aug 29 2022 12:56:00 GMT+0…",""" <p>Hi dokter, dok saya ada g…",""" <p>Selamat siang, terima kas…"
"""62dba6259bff7407087e7fd6""","""Tindakan untuk mengatasi kerus…","""Sat Jul 23 2022 14:41:00 GMT+0…","""Sat Jul 23 2022 16:25:00 GMT+0…",""" <p>Alo dokter... Saya ingin …",""" <p>Alo, terimakasih atas per…"
"""625abb4407faa10613bb6bf7""","""Apakah ablasi retina yang di s…","""Sat Apr 16 2022 19:49:00 GMT+0…","""Sat Apr 16 2022 21:06:00 GMT+0…",""" <p>Selamat malam dok, saya m…",""" <p>Terima kasih atas pertany…"
"""6255666503dcb1060dc9dfe9""","""Penyebab dan penanganan ablasi…","""Tue Apr 12 2022 18:45:00 GMT+0…","""Tue Apr 12 2022 19:22:00 GMT+0…",""" <p>mohon maaf dok mau tanya …",""" <p>Alo, terima kasih ya suda…"


In [20]:
date_sample = (
    df
    .select([
        "question_post_date",
        "answer_post_date"
    ])
    .head(20)
    .collect()
)

date_sample

question_post_date,answer_post_date
str,str
"""Fri Jul 14 2023 13:25:00 GMT+0…","""Fri Jul 14 2023 17:59:00 GMT+0…"
"""Mon Aug 29 2022 08:25:00 GMT+0…","""Mon Aug 29 2022 12:56:00 GMT+0…"
"""Sat Jul 23 2022 14:41:00 GMT+0…","""Sat Jul 23 2022 16:25:00 GMT+0…"
"""Sat Apr 16 2022 19:49:00 GMT+0…","""Sat Apr 16 2022 21:06:00 GMT+0…"
"""Tue Apr 12 2022 18:45:00 GMT+0…","""Tue Apr 12 2022 19:22:00 GMT+0…"
…,…
"""Mon Jul 19 2021 22:12:00 GMT+0…","""Tue Jul 20 2021 10:55:00 GMT+0…"
"""Mon Jul 05 2021 15:56:00 GMT+0…","""Mon Jul 05 2021 23:24:00 GMT+0…"
"""Wed Jun 16 2021 08:18:00 GMT+0…","""Wed Jun 16 2021 12:23:00 GMT+0…"
